In [ ]:
from google.colab import drive
drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)

Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/SnowPole_Detection_Dataset/
# !git clone https://github.com/ultralytics/ultralytics.git

/content/drive/.shortcut-targets-by-id/1ofcfwBISHeUE4V1x6Bj-J5Mw3-QSUHFB/SnowPole_Detection_Dataset


In [ ]:
!pip uninstall -y ultralytics

In [ ]:
!rm -rf /content/ultralytics
!git clone https://github.com/MuhammadIbneRafiq/ultralytics4channel /content/ultralytics

Cloning into '/content/ultralytics'...
remote: Enumerating objects: 276, done.
remote: Counting objects: 100% (276/276), done.
remote: Compressing objects: 100% (222/222), done.
remote: Total 276 (delta 65), reused 254 (delta 52), pack-reused 0 (from 0)
Receiving objects: 100% (276/276), 706.61 KiB | 1.57 MiB/s, done.
Resolving deltas: 100% (65/65), done.


In [ ]:
!pip install -q ultralytics==8.2.5

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.0/755.0 kB 53.9 MB/s eta 0:00:00


In [ ]:
import sys
sys.path.insert(0, "/content")  # parent of the ultralytics package

import ultralytics
from ultralytics import YOLO

print("Ultralytics module file:", ultralytics.__file__)


Ultralytics module file: /content/ultralytics/__init__.py


In [ ]:
import torch
from ultralytics import YOLO
from pathlib import Path
import shutil
import cv2
import numpy as np
import yaml
from tqdm import tqdm

from ultralytics import YOLO
import torch

from torch.utils.data import Dataset, DataLoader

In [ ]:
COMB_ROOT  = Path("/content/drive/MyDrive/SnowPole_Detection_Dataset/images")

# 1-channel range-normalized images
RANGE_ROOT = Path("/content/drive/MyDrive/SnowPole_Detection_Dataset/range-normalized-continuous")

# New 4-channel dual-input dataset
DUAL_ROOT  = Path("/content/drive/MyDrive/SnowPole_Detection_Dataset/comb4-range-signal-reflec_and_range")
DUAL_ROOT.mkdir(parents=True, exist_ok=True)

print("COMB_ROOT :", COMB_ROOT)
print("RANGE_ROOT:", RANGE_ROOT)
print("DUAL_ROOT :", DUAL_ROOT)

COMB_ROOT : /content/drive/MyDrive/SnowPole_Detection_Dataset/images
RANGE_ROOT: /content/drive/MyDrive/SnowPole_Detection_Dataset/range-normalized-continuous
DUAL_ROOT : /content/drive/MyDrive/SnowPole_Detection_Dataset/comb4-range-signal-reflec_and_range


In [ ]:
# !yolo train model=yolov9t.pt epochs=150 imgsz=1024 device=0 batch=2 data=/content/drive/MyDrive/data.yaml project=/content/drive/MyDrive/SnowPole_Detection_Dataset/comb4-range-signal-reflec

In [ ]:
# !yolo val \
#   model=/content/drive/.shortcut-targets-by-id/1ofcfwBISHeUE4V1x6Bj-J5Mw3-QSUHFB/SnowPole_Detection_Dataset/comb4-range-signal-reflec-v11n/train2/weights/best.pt \
#   data=/content/drive/MyDrive/data.yaml \
#   split=test \
#   imgsz=1024 \
#   device=0 \
#   batch=16 \
#   project="comb5_signal_reflec_range_11n" \
#   name="comb5_signal_reflec_range_11n_test_eval"


In [ ]:
def make_dual_split_npy(split: str,
                        comb_root: Path,
                        range_root: Path,
                        dual_root: Path,
                        save_as_png: bool = True):
    """
    Create 4-channel dual images by stacking comb RGB (BGR) + range (.npy float32 [0,1]).
    - comb_root: root containing comb_root/images/<split>/*.png and comb_root/labels/<split>/*.txt
    - range_root: root containing range_root/<split>/*.npy (each named like the comb image stem)
    - dual_root: destination root; will create dual_root/images/<split> and dual_root/labels/<split>
    - save_as_png: if True, save stacked RGBA PNGs (4 channel) so existing YOLO loaders can read them.
                   (range channel is quantized to uint8 for the PNG; the original .npy is left unchanged)
    """
    comb_img_dir   = comb_root / split
    src_lbl_dir    = comb_root / "../" / "labels" / split
    range_npy_dir  = range_root / split
    dual_img_dir   = dual_root / "images" / split
    dual_lbl_dir   = dual_root / "labels" / split

    # if this split already has images, skip doing anything
    if dual_img_dir.exists() and any(dual_img_dir.glob("*.png")):
        print(f"[{split}] dual images already exist in {dual_img_dir}, skipping.")
        return

    dual_img_dir.mkdir(parents=True, exist_ok=True)
    dual_lbl_dir.mkdir(parents=True, exist_ok=True)

    # copy labels from comb labels to dual labels (they are the same)
    if src_lbl_dir.exists():
        for lbl in src_lbl_dir.glob("*.txt"):
            # copy2 preserves timestamps, etc.
            shutil.copy2(lbl, dual_lbl_dir / lbl.name)
    else:
        print(f"Warning: source label dir not found: {src_lbl_dir}")

    # gather comb images
    img_files = sorted(comb_img_dir.glob("*.*"))
    print(f"[{split}] comb images found: {len(img_files)}")

    for comb_path in tqdm(img_files, desc=f"make_dual_split ({split})"):
        stem = comb_path.stem

        # read comb RGB (OpenCV: BGR)
        comb = cv2.imread(str(comb_path), cv2.IMREAD_COLOR)
        if comb is None:
            print("Could not read comb image:", comb_path)
            continue

        # read corresponding range .npy
        range_npy_path = range_npy_dir / f"{stem}.npy"
        if not range_npy_path.exists():
            # try alternative stem patterns if needed
            print(f"Missing range .npy for {stem} -> {range_npy_path} (skipping)")
            continue

        try:
            range_arr = np.load(str(range_npy_path))   # expected float32 in [0,1]
        except Exception as e:
            print(f"Failed to load {range_npy_path}: {e}")
            continue

        # squeeze any extra dims
        if range_arr.ndim == 3 and range_arr.shape[0] in (1,):
            range_arr = np.squeeze(range_arr, axis=0)
        if range_arr.ndim != 2:
            # if it has a channel dim (H,W,1) -> squeeze
            if range_arr.ndim == 3 and range_arr.shape[2] == 1:
                range_arr = np.squeeze(range_arr, axis=2)
            else:
                print(f"Unexpected shape for range npy {range_npy_path}: {range_arr.shape} (skipping)")
                continue

        # ensure float32 and clip to [0,1]
        range_arr = range_arr.astype(np.float32)
        range_arr = np.clip(range_arr, 0.0, 1.0)

        # convert range to uint8 for stacking if saving PNGs (visualization/training with standard loader)
        range_uint8 = (range_arr * 255.0).astype(np.uint8)

        # resize range to comb dims if necessary (note cv2 resize expects (width, height))
        if range_uint8.shape != comb.shape[:2]:
            range_uint8 = cv2.resize(range_uint8, (comb.shape[1], comb.shape[0]), interpolation=cv2.INTER_NEAREST)

        # stack into 4-channel: B, G, R, RANGE
        rgba = np.dstack([comb, range_uint8])  # result dtype uint8, shape (H, W, 4)

        # write stacked 4-channel PNG so YOLO-like image loaders can ingest it
        if save_as_png:
            out_img_path = dual_img_dir / f"{stem}.png"
            # OpenCV will write all 4 channels to PNG when given a 4-channel array.
            cv2.imwrite(str(out_img_path), rgba)

    print(f"[{split}] done. Dual images written to {dual_img_dir}, labels copied to {dual_lbl_dir}")


# Run for all splits (call this cell)
for split in ["train", "valid", "test"]:
    make_dual_split_npy(split, comb_root=COMB_ROOT, range_root=RANGE_ROOT, dual_root=DUAL_ROOT, save_as_png=True)


[train] dual images already exist in /content/drive/MyDrive/SnowPole_Detection_Dataset/comb4-range-signal-reflec_and_range/images/train, skipping.
[valid] dual images already exist in /content/drive/MyDrive/SnowPole_Detection_Dataset/comb4-range-signal-reflec_and_range/images/valid, skipping.
[test] dual images already exist in /content/drive/MyDrive/SnowPole_Detection_Dataset/comb4-range-signal-reflec_and_range/images/test, skipping.


In [ ]:
ORIG_DATA_YAML = COMB_ROOT / "../" /"data.yaml"
DUAL_DATA_YAML = DUAL_ROOT / "data.yaml"

# with open(ORIG_DATA_YAML, "r") as f:
#     cfg = yaml.safe_load(f)

# base = DUAL_ROOT

# def make_rel(p):
#     # p might be absolute or relative – we point to new dual root
#     p = Path(p)
#     return str((base / "images" / p.name).parent)  # keep split names

# # If your original yaml used explicit paths, you can instead do:
# # cfg["path"]  = str(DUAL_ROOT)
# cfg["path"]  = str(DUAL_ROOT)
# cfg["train"] = "images/train"
# cfg["valid"]   = "images/valid"
# cfg["test"]  = "images/test"
# cfg["channels"] = 4          # tell YOLO this is 4-channel data with the RGB-Alpha

# with open(DUAL_DATA_YAML, "w") as f:
#     yaml.safe_dump(cfg, f)

# print(DUAL_DATA_YAML.read_text())

In [ ]:
from ultralytics import YOLO
import torch
from ultralytics.nn.tasks import DetectionModel

import torch
from ultralytics.nn.tasks import DetectionModel
# from ultralytics.nn.modules.block import ELAN1, AConv, RepNCSPELAN4, SPPELAN

# allow the DetectionModel class for unpickling

torch.serialization.add_safe_globals([
    DetectionModel,
    # ELAN1,
    # AConv,
    # RepNCSPELAN4,
    # SPPELAN,
])


model = YOLO("yolov8n.pt")  # or YOLO("yolov9t.pt", task="detect")
model.model.eval()

model.train(
    data=str(DUAL_DATA_YAML),
    epochs=400,
    imgsz=1024,
    device=0,
    amp=False,  # Disable AMP

    batch=16,
    project="dual_comb_range_experiments",
    name="dual_comb_rgb_plus_range_v9t_4ch_825",
)


New https://pypi.org/project/ultralytics/8.3.241 available 😃 Update with 'pip install -U ultralytics'
Ultralytics YOLOv8.2.5 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (NVIDIA A100-SXM4-80GB, 81222MiB)
engine/trainer: task=detect, mode=train, model=yolov8n.pt, data=/content/drive/MyDrive/SnowPole_Detection_Dataset/comb4-range-signal-reflec_and_range/data.yaml, epochs=400, time=None, patience=100, batch=16, imgsz=1024, save=True, save_period=-1, cache=False, device=0, workers=8, project=dual_comb_range_experiments, name=dual_comb_rgb_plus_range_v9t_4ch_82510, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=False, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, 

train: Scanning /content/drive/.shortcut-targets-by-id/1ofcfwBISHeUE4V1x6Bj-J5Mw3-QSUHFB/SnowPole_Detection_Dataset/comb4-range-signal-reflec_and_range/labels/train.cache... 1367 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1367/1367 [00:00<?, ?it/s]

albumentations: 



/content/ultralytics/data/augment.py:891: UserWarning: Argument(s) 'quality_lower' are not valid for transform ImageCompression
  A.ImageCompression(quality_lower=75, p=0.0),
/usr/local/lib/python3.12/dist-packages/albumentations/core/composition.py:331: UserWarning: Got processor for bboxes, but no transform to process it.
  self._set_keys()
val: Scanning /content/drive/.shortcut-targets-by-id/1ofcfwBISHeUE4V1x6Bj-J5Mw3-QSUHFB/SnowPole_Detection_Dataset/comb4-range-signal-reflec_and_range/labels/valid.cache... 390 images, 0 backgrounds, 0 corrupt: 100%|██████████| 390/390 [00:00<?, ?it/s]


Plotting labels to dual_comb_range_experiments/dual_comb_rgb_plus_range_v9t_4ch_82510/labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 57 weight(decay=0.0), 64 weight(decay=0.0005), 63 bias(decay=0.0)
TensorBoard: model graph visualization added ✅
Image sizes 1024 train, 1024 val
Using 8 dataloader workers
Logging results to dual_comb_range_experiments/dual_comb_rgb_plus_range_v9t_4ch_82510
Starting training for 400 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


...

EarlyStopping: Training stopped early as no improvement observed in last 100 epochs. Best results observed at epoch 134, best model saved as best.pt.
To update EarlyStopping(patience=100) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

234 epochs completed in 1.026 hours.
Optimizer stripped from dual_comb_range_experiments/dual_comb_rgb_plus_range_v9t_4ch_82510/weights/last.pt, 6.2MB
Optimizer stripped from dual_comb_range_experiments/dual_comb_rgb_plus_range_v9t_4ch_82510/weights/best.pt, 6.2MB

Validating dual_comb_range_experiments/dual_comb_rgb_plus_range_v9t_4ch_82510/weights/best.pt...
Ultralytics YOLOv8.2.5 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (NVIDIA A100-SXM4-80GB, 81222MiB)
Model summary (fused): 168 layers, 3005987 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 13/13 [00:02<00:00,  6.19it/s]


                   all        390        789      0.876      0.837      0.882      0.436
Speed: 0.1ms preprocess, 0.4ms inference, 0.0ms loss, 1.0ms postprocess per image
Results saved to dual_comb_range_experiments/dual_comb_rgb_plus_range_v9t_4ch_82510


lr/pg0,████▇▇▇▇▆▆▆▆▆▆▆▆▅▅▅▅▄▄▃▃▃▃▃▃▃▃▃▃▂▂▂▂▁▁▁▁
lr/pg1,▁████▇▇▇▇▇▇▇▇▇▇▆▆▆▅▅▅▅▅▅▅▅▄▄▃▃▃▃▃▃▃▃▂▂▂▂
lr/pg2,█████▇▇▇▇▇▆▆▆▆▆▆▆▅▅▅▅▅▅▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁
metrics/mAP50(B),▁▄▆▅▆▆▆▆▆▇▇▇█▇████▇██▇██████████████████
metrics/mAP50-95(B),▃▂▄▅▁▄▄▄▂▆▄▃▆▆▇▇▇▇▇█████████▇▇███▇▇▇████
metrics/precision(B),▁▆▇▆▆▇▆▆▇▅▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇██▇▆▇▇▇▇█▇▇▆█▇▇
metrics/recall(B),▁▇▇▇▇▇▇▇▇▇▇▇▇█▇█████▇███████████████████
model/GFLOPs,▁
model/parameters,▁
model/speed_PyTorch(ms),▁
+6,...


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7ce6181b18b0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [ ]:
from ultralytics import YOLO

# if still in memory
# model = model

# or reload from best checkpoint
# model = YOLO("path/to/runs/detect/dual_comb_rgb_plus_range_v8n_4ch/weights/best.pt")

metrics = model.val(
    data=str(DUAL_DATA_YAML),  # your data.yaml with test path
    split="test",              # use test set instead of val
    imgsz=1024,
    device=0,                # GPU id, or "cpu"
    batch=16,
    project="dual_comb_range_experiments",
    name="dual_comb_rgb_plus_range_v8n_4ch_test",
)

print(metrics)  # mAP50, mAP50-95, precision, recall, etc.


Ultralytics YOLOv8.2.5 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (NVIDIA A100-SXM4-80GB, 81222MiB)
Model summary (fused): 168 layers, 3005987 parameters, 0 gradients, 8.1 GFLOPs


val: Scanning /content/drive/.shortcut-targets-by-id/1ofcfwBISHeUE4V1x6Bj-J5Mw3-QSUHFB/SnowPole_Detection_Dataset/comb4-range-signal-reflec_and_range/labels/test... 197 images, 0 backgrounds, 0 corrupt: 100%|██████████| 197/197 [00:51<00:00,  3.81it/s]

val: New cache created: /content/drive/.shortcut-targets-by-id/1ofcfwBISHeUE4V1x6Bj-J5Mw3-QSUHFB/SnowPole_Detection_Dataset/comb4-range-signal-reflec_and_range/labels/test.cache



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 13/13 [00:02<00:00,  6.29it/s]


                   all        197        395      0.857      0.834      0.883      0.449
Speed: 0.1ms preprocess, 1.6ms inference, 0.0ms loss, 1.2ms postprocess per image
Results saved to dual_comb_range_experiments/dual_comb_rgb_plus_range_v8n_4ch_test
ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7ce66060ff80>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029, 